# S43_08 — Google Cloud Platform: Vertex AI

## Overview

**Vertex AI** is Google Cloud's unified ML platform. It consolidates what was previously scattered across Cloud ML Engine, AutoML, and AI Platform into a single product. It has the tightest integration with **BigQuery** (for data at scale), **Google's foundation models** (Gemini, Imagen), and **Kubeflow Pipelines** for ML orchestration.

Vertex AI is the natural choice in GCP-native organisations and is particularly strong when your data already lives in BigQuery or GCS.

## Core concepts

| Concept | Vertex AI term | AWS SageMaker equivalent |
|---------|---------------|-------------------------|
| Project namespace | **Project + Region** | Domain |
| Compute for training | **Custom Training Job** | Training job |
| Experiment tracking | **Vertex AI Experiments** (MLflow-compatible) | SageMaker Experiments |
| Model storage | **Model Registry** | Model Registry |
| Serving endpoint | **Endpoint** | SageMaker Endpoint |
| ML pipeline | **Vertex AI Pipelines** (Kubeflow-based) | SageMaker Pipelines |
| AutoML | **AutoML** | SageMaker Autopilot |
| Notebook environment | **Workbench** | SageMaker Studio |
| Foundation models | **Model Garden** (Gemini, Imagen, Codey) | Bedrock (Claude, Llama, etc.) |

## Installation

In [ ]:
# pip install google-cloud-aiplatform
import vertexai
from google.cloud import aiplatform

# Initialise with your project and region
vertexai.init(project='your-gcp-project', location='us-central1')
aiplatform.init(project='your-gcp-project', location='us-central1')

## Submitting a custom training job

In [ ]:
from google.cloud import aiplatform

# Submit a custom training job using a pre-built container
job = aiplatform.CustomTrainingJob(
    display_name='sklearn-breast-cancer-rf',
    script_path='train.py',                         # your local training script
    container_uri='us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-0:latest',
    requirements=['scikit-learn', 'mlflow'],
    model_serving_container_image_uri=(
        'us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest'
    ),
)

# model = job.run(
#     machine_type='n1-standard-4',
#     replica_count=1,
# )

## Experiment tracking

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Vertex AI Experiments are MLflow-compatible
# mlflow.set_tracking_uri(aiplatform.get_experiment('my-experiment').resource_name)

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

with mlflow.start_run(run_name='rf-vertex-demo'):
    n_estimators = 100
    mlflow.log_param('n_estimators', n_estimators)

    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric('accuracy', acc)
    mlflow.sklearn.log_model(model, artifact_path='model')

    print(f'Accuracy: {acc:.4f}')

## Deploying a model to an endpoint

In [ ]:
from google.cloud import aiplatform

# Upload model to registry
# model = aiplatform.Model.upload(
#     display_name='breast-cancer-rf',
#     artifact_uri='gs://your-bucket/model/',    # GCS path
#     serving_container_image_uri=(
#         'us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest'
#     ),
# )

# Create endpoint and deploy
# endpoint = aiplatform.Endpoint.create(display_name='breast-cancer-endpoint')
# model.deploy(
#     endpoint=endpoint,
#     machine_type='n1-standard-2',
#     min_replica_count=1,
#     max_replica_count=3,   # auto-scales
# )

# Predict
# predictions = endpoint.predict(instances=X_test[:5].tolist())
# print(predictions.predictions)

## Vertex AI Pipelines (Kubeflow-based)

Vertex AI Pipelines use the **Kubeflow Pipelines (KFP) SDK** to define and orchestrate multi-step ML workflows.

In [ ]:
# pip install kfp google-cloud-pipeline-components
from kfp import dsl
from kfp.v2 import compiler

@dsl.component(base_image='python:3.11', packages_to_install=['scikit-learn'])
def train_model(n_estimators: int) -> float:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))

@dsl.pipeline(name='breast-cancer-pipeline')
def my_pipeline(n_estimators: int = 100):
    train_task = train_model(n_estimators=n_estimators)

# compiler.Compiler().compile(my_pipeline, 'pipeline.json')
# job = aiplatform.PipelineJob(display_name='demo', template_path='pipeline.json')
# job.run()

## Using Google Foundation Models (Gemini)

Vertex AI's **Model Garden** provides access to Google's Gemini models for text, code, and multimodal tasks.

In [ ]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project='your-gcp-project', location='us-central1')

model = GenerativeModel('gemini-1.5-flash')  # or gemini-1.5-pro

response = model.generate_content(
    'Explain gradient boosting in two sentences.'
)
print(response.text)

## Vertex AI vs AWS SageMaker vs Azure ML

| | Vertex AI (GCP) | SageMaker (AWS) | Azure ML |
|---|---|---|---|
| Pipeline SDK | Kubeflow Pipelines (KFP) | Proprietary | Azure ML Pipelines |
| Experiment tracking | MLflow-compatible | SageMaker Experiments | MLflow (native) |
| Foundation models | Gemini, Imagen (Model Garden) | Claude, Llama (Bedrock) | OpenAI, Phi (Azure OpenAI) |
| AutoML | AutoML Tables/Vision/NLP | Autopilot | AutoML |
| Data integration | BigQuery-native | S3/Glue | Azure Data Lake / Synapse |
| Notebook environment | Workbench (managed JupyterLab) | Studio | Compute Instances |
| Hybrid/edge | Vertex AI Edge | SageMaker Edge | Azure Arc |

## Further reading
- [Vertex AI documentation](https://cloud.google.com/vertex-ai/docs)
- [Vertex AI Python SDK](https://cloud.google.com/python/docs/reference/aiplatform/latest)
- [Kubeflow Pipelines SDK](https://www.kubeflow.org/docs/components/pipelines/)
- [Gemini API in Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/start/quickstarts)
